# Videos with original FPS

In [ ]:
from pathlib import Path
import subprocess
import shlex
import re
import time

# ---------------- Configuration ----------------
ROOT_DIR = Path(r"A:\rawData\raw_observationSessions\ECG_Controls")

# IMPORTANT: this is an absolute path, so don't join it onto ROOT_DIR
OUTPUT_ROOT = Path(r"C:\Users\stagk\Desktop\TempECG")

FFMPEG = r"C:\Users\stagk\Desktop\ffmpeg\bin\ffmpeg.exe"  # or "ffmpeg" if in PATH
PRESET = "slow"
CRF = 22
OVERWRITE = True
DRY_RUN = False
CUT_TO = "00:20:05"

# Bento/MATLAB compatibility + FPS normalization
BENTO_PIX_FMT = "yuv420p"              # key: avoid yuv444p / 4:4:4
FORCE_50FPS_TAGS = {"face", "mouseView", "topView"}
TARGET_FPS = 50

# NEW: per-tag CRF override(s)
CRF_BY_TAG = {
    "mouseView": 20,   # mouseView gets higher quality (lower CRF)
}
# ------------------------------------------------


def find_mouse_id(mouse_folder_name: str) -> str:
    """Extract mouseID from folder like '20250817_mouse975826'."""
    m = re.search(r"_(mouse\d+)", mouse_folder_name, flags=re.IGNORECASE)
    if m:
        return m.group(1)
    m2 = re.search(r"(\d{5,})", mouse_folder_name)
    if m2:
        return f"mouse{m2.group(1)}"
    return mouse_folder_name


BEHAVIOR_TAGS = {
    "arena (mouse view)": "mouseView",
    "arena (top view)": "topView",
    "face": "face",
    "hollywoodscene": "hs",
}


def detect_behavior_tag(path: Path) -> str | None:
    for p in [path.parent] + list(path.parents):
        name = p.name.strip().lower()
        name_norm = re.sub(r"\s+", " ", name)
        if name_norm in BEHAVIOR_TAGS:
            return BEHAVIOR_TAGS[name_norm]
        if "mouse view" in name or "arena (mouse view)" in name_norm:
            return "mouseView"
        if "top view" in name or "arena (top view)" in name_norm:
            return "topView"
        if name == "face":
            return "face"
        if "hollywood" in name:
            return "hs"
    return None


def is_inside_arena_folder(path: Path) -> bool:
    for p in [path.parent] + list(path.parents):
        if "arena" in p.name.lower():
            return True
    return False


def should_skip_input(src_path: Path, output_root: Path) -> bool:
    """
    Skip:
    - already compressed inputs (_comp.avi)
    - files inside the output tree
    """
    if src_path.name.lower().endswith("_comp.avi"):
        return True
    try:
        src_path.relative_to(output_root)
        return True
    except Exception:
        return False


def output_exists(dst: Path) -> bool:
    return dst.exists()


def compress_with_ffmpeg(
    src: Path,
    dst: Path,
    tag: str,
    preset: str = PRESET,
    crf: int = CRF,
    overwrite: bool = OVERWRITE,
    cut_to: str = CUT_TO,
    flip_h: bool = False,
):
    """Run ffmpeg compression and report timing."""

    # NEW: override CRF for specific tags
    effective_crf = CRF_BY_TAG.get(tag, crf)

    # Build a single filter chain (fps + optional hflip)
    vf_filters = []

    # Force constant 50 fps only for face/mouseView/topView (hs keeps native fps)
    if tag in FORCE_50FPS_TAGS:
        vf_filters.append(f"fps={TARGET_FPS}")

    if flip_h:
        vf_filters.append("hflip")

    cmd = [
        FFMPEG,
        "-i", str(src),
        "-c:v", "libx264",
        "-preset", preset,
        "-crf", str(effective_crf),

        # KEY: enforce MATLAB-friendly pixel format (avoid yuv444p)
        "-pix_fmt", BENTO_PIX_FMT,

        "-to", str(cut_to),

        # drop audio
        "-an",
    ]

    # Apply filters if needed
    if vf_filters:
        cmd += ["-vf", ",".join(vf_filters)]
        # If we're forcing fps, keep output CFR to avoid timestamp weirdness
        if tag in FORCE_50FPS_TAGS:
            cmd += ["-vsync", "cfr"]

    if overwrite:
        cmd.append("-y")
    cmd.append(str(dst))

    cmd_str = " ".join(shlex.quote(c) for c in cmd)
    print(f"\n[COMMAND] {cmd_str}")

    if DRY_RUN:
        print("[DRY RUN] Skipping execution.")
        return

    start = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.time() - start

    if result.returncode != 0:
        print(f"[ERROR] ffmpeg failed for {src.name}")
        print(result.stderr)
    else:
        notes = []
        if tag in FORCE_50FPS_TAGS:
            notes.append(f"fps={TARGET_FPS}")
        if flip_h:
            notes.append("hflip")
        note_str = (" | " + ", ".join(notes)) if notes else ""
        print(f"[OK] {dst.name} | CRF={effective_crf}{note_str} | Time: {elapsed/60:.2f} min ({elapsed:.1f} s)")


def main():
    output_root = OUTPUT_ROOT
    output_root.mkdir(parents=True, exist_ok=True)

    total = skipped = processed = 0

    # Iterate mouse folders (exclude output root if it happens to be under ROOT_DIR)
    for mouse_dir in [p for p in ROOT_DIR.iterdir() if p.is_dir()]:
        try:
            if mouse_dir.resolve() == output_root.resolve():
                continue
        except Exception:
            pass

        mouse_id = find_mouse_id(mouse_dir.name)

        for day_dir in [d for d in mouse_dir.glob("Day*") if d.is_dir()]:
            day = day_dir.name

            for avi in day_dir.rglob("*.avi"):
                total += 1

                if should_skip_input(avi, output_root):
                    skipped += 1
                    continue

                tag = detect_behavior_tag(avi)
                if tag is None:
                    print(f"[WARN] Could not determine behavior tag for: {avi}")
                    skipped += 1
                    continue

                out_dir = output_root / mouse_dir.name / day
                out_dir.mkdir(parents=True, exist_ok=True)

                out_name = f"{mouse_id}_{day}_{tag}_comp.avi"
                out_path = out_dir / out_name

                if output_exists(out_path):
                    print(f"[SKIP existing] {out_path}")
                    skipped += 1
                    continue

                flip_h = is_inside_arena_folder(avi)

                compress_with_ffmpeg(avi, out_path, tag=tag, flip_h=flip_h)
                processed += 1

    print("\n=== Summary ===")
    print(f"Total .avi files found : {total}")
    print(f"Skipped                 : {skipped}")
    print(f"Compressed              : {processed}")


if __name__ == "__main__":
    main()



[COMMAND] 'C:\Users\stagk\Desktop\ffmpeg\bin\ffmpeg.exe' -i 'A:\rawData\raw_observationSessions\ECG_Controls\20261905_mouse1070002\Day01\behavioralData\Arena (Mouse View)\19-15-55.avi' -c:v libx264 -preset slow -crf 20 -pix_fmt yuv420p -to 00:20:05 -an -vf fps=50,hflip -vsync cfr -y 'C:\Users\stagk\Desktop\TempECG\20261905_mouse1070002\Day01\mouse1070002_Day01_mouseView_comp.avi'
[OK] mouse1070002_Day01_mouseView_comp.avi | CRF=20 | fps=50, hflip | Time: 4.55 min (272.8 s)

[COMMAND] 'C:\Users\stagk\Desktop\ffmpeg\bin\ffmpeg.exe' -i 'A:\rawData\raw_observationSessions\ECG_Controls\20261905_mouse1070002\Day01\behavioralData\Arena (Top view)\19-15-55.avi' -c:v libx264 -preset slow -crf 22 -pix_fmt yuv420p -to 00:20:05 -an -vf fps=50,hflip -vsync cfr -y 'C:\Users\stagk\Desktop\TempECG\20261905_mouse1070002\Day01\mouse1070002_Day01_topView_comp.avi'
[OK] mouse1070002_Day01_topView_comp.avi | CRF=22 | fps=50, hflip | Time: 5.61 min (336.7 s)

[COMMAND] 'C:\Users\stagk\Desktop\ffmpeg\bin\ff

# Bento ready videos, all at 50 FPS

In [ ]:
from pathlib import Path
import subprocess
import shlex
import re
import time


# ---------------- Configuration ----------------

ROOT_DIR = Path(r"D:\trainingAggression_MR")

# IMPORTANT: absolute path; do not join onto ROOT_DIR
OUTPUT_ROOT = Path(r"D:\observationSessions_bento")

FFMPEG = r"C:\Users\stagk\Desktop\ffmpeg\bin\ffmpeg.exe"
# Or simply:
# FFMPEG = "ffmpeg"
# if ffmpeg is available in PATH

PRESET = "slow"
CRF = 22

OVERWRITE = True
DRY_RUN = False

CUT_TO = "00:20:05"

# Bento / MATLAB compatibility
BENTO_PIX_FMT = "yuv420p"

# FPS normalization
FORCE_50FPS_TAGS = {
    "face",
    "mouseView",
    "topView",
}

TARGET_FPS = 50

# Per-camera CRF overrides
CRF_BY_TAG = {
    "mouseView": 20,
}

# ------------------------------------------------


BEHAVIOR_TAGS = {
    "arena (mouse view)": "mouseView",
    "arena (top view)": "topView",
    "face": "face",
    "hollywoodscene": "hs",
}


def find_mouse_id(mouse_folder_name: str) -> str:
    """
    Extract mouse ID from names such as:

        20250817_mouse975826
        mouse975826

    Returns:
        mouse975826
    """

    m = re.search(
        r"(mouse\d+)",
        mouse_folder_name,
        flags=re.IGNORECASE,
    )

    if m:
        return m.group(1).lower()

    # Fallback: find a reasonably long number
    m2 = re.search(r"(\d{5,})", mouse_folder_name)

    if m2:
        return f"mouse{m2.group(1)}"

    return mouse_folder_name


def find_mouse_folder(path: Path, root_dir: Path) -> Path | None:
    """
    Search upward from an AVI file and find the nearest ancestor
    whose folder name contains something like 'mouse975826'.

    This makes mouse-folder detection recursive and does not require
    the mouse folder to be directly underneath ROOT_DIR.
    """

    for parent in path.parents:

        # Stop once we move outside ROOT_DIR
        try:
            parent.relative_to(root_dir)
        except ValueError:
            break

        if re.search(
            r"mouse\d+",
            parent.name,
            flags=re.IGNORECASE,
        ):
            return parent

    return None


def find_day_folder(path: Path, root_dir: Path) -> Path | None:
    """
    Search upward from an AVI file for the nearest folder beginning
    with 'Day'.

    Examples:
        Day1
        Day2
        Day10
        Day_3
    """

    for parent in path.parents:

        try:
            parent.relative_to(root_dir)
        except ValueError:
            break

        if re.match(
            r"^day",
            parent.name,
            flags=re.IGNORECASE,
        ):
            return parent

    return None


def detect_behavior_tag(path: Path) -> str | None:
    """
    Determine which behavior/camera type an AVI belongs to by
    examining its parent folders.
    """

    for parent in [path.parent] + list(path.parents):

        name = parent.name.strip().lower()
        name_norm = re.sub(r"\s+", " ", name)

        # Exact known folder names
        if name_norm in BEHAVIOR_TAGS:
            return BEHAVIOR_TAGS[name_norm]

        # More permissive matching
        if "mouse view" in name_norm:
            return "mouseView"

        if "top view" in name_norm:
            return "topView"

        if name_norm == "face":
            return "face"

        if "hollywood" in name_norm:
            return "hs"

    return None


def is_inside_arena_folder(path: Path) -> bool:
    """
    Return True if the AVI is somewhere underneath a folder whose
    name contains 'arena'.
    """

    for parent in [path.parent] + list(path.parents):

        if "arena" in parent.name.lower():
            return True

    return False


def should_skip_input(
    src_path: Path,
    output_root: Path,
) -> bool:
    """
    Skip:

    - already compressed files ending in _comp.avi
    - files that are already inside OUTPUT_ROOT
    """

    if src_path.name.lower().endswith("_comp.avi"):
        return True

    try:
        src_path.resolve().relative_to(output_root.resolve())
        return True
    except ValueError:
        pass
    except OSError:
        pass

    return False


def output_exists(dst: Path) -> bool:
    """Check whether the output file already exists."""

    return dst.exists()


def compress_with_ffmpeg(
    src: Path,
    dst: Path,
    tag: str,
    preset: str = PRESET,
    crf: int = CRF,
    overwrite: bool = OVERWRITE,
    cut_to: str = CUT_TO,
    flip_h: bool = False,
):
    """
    Compress one AVI file with ffmpeg.

    - H.264 / libx264
    - MATLAB/Bento-friendly yuv420p
    - selected cameras forced to 50 fps CFR
    - arena recordings horizontally flipped
    - audio removed
    - video cut at CUT_TO
    """

    effective_crf = CRF_BY_TAG.get(tag, crf)

    # ------------------------------------------------------------
    # Build video filter chain
    # ------------------------------------------------------------

    vf_filters = []

    if tag in FORCE_50FPS_TAGS:
        vf_filters.append(f"fps={TARGET_FPS}")

    if flip_h:
        vf_filters.append("hflip")

    # ------------------------------------------------------------
    # Build ffmpeg command
    # ------------------------------------------------------------

    cmd = [
        FFMPEG,
        "-i",
        str(src),

        "-c:v",
        "libx264",

        "-preset",
        preset,

        "-crf",
        str(effective_crf),

        "-pix_fmt",
        BENTO_PIX_FMT,

        "-to",
        str(cut_to),

        # Remove audio
        "-an",
    ]

    # Add filters
    if vf_filters:
        cmd += [
            "-vf",
            ",".join(vf_filters),
        ]

    # Force CFR when normalizing FPS
    if tag in FORCE_50FPS_TAGS:
        cmd += [
            "-vsync",
            "cfr",
        ]

    # Overwrite existing files if requested
    if overwrite:
        cmd.append("-y")
    else:
        cmd.append("-n")

    cmd.append(str(dst))

    # ------------------------------------------------------------
    # Display command
    # ------------------------------------------------------------

    cmd_str = " ".join(
        shlex.quote(str(c))
        for c in cmd
    )

    print()
    print(f"[COMMAND] {cmd_str}")

    if DRY_RUN:
        print("[DRY RUN] Skipping execution.")
        return True

    # ------------------------------------------------------------
    # Execute ffmpeg
    # ------------------------------------------------------------

    start = time.time()

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
    )

    elapsed = time.time() - start

    if result.returncode != 0:

        print()
        print(f"[ERROR] ffmpeg failed for:")
        print(f"        {src}")
        print()
        print(result.stderr)

        return False

    # ------------------------------------------------------------
    # Success message
    # ------------------------------------------------------------

    notes = []

    if tag in FORCE_50FPS_TAGS:
        notes.append(f"fps={TARGET_FPS}")

    if flip_h:
        notes.append("hflip")

    note_str = ""

    if notes:
        note_str = " | " + ", ".join(notes)

    print(
        f"[OK] {dst.name}"
        f" | CRF={effective_crf}"
        f"{note_str}"
        f" | Time: {elapsed / 60:.2f} min"
        f" ({elapsed:.1f} s)"
    )

    return True


def main():

    # ------------------------------------------------------------
    # Validate directories
    # ------------------------------------------------------------

    if not ROOT_DIR.exists():
        print(f"[ERROR] ROOT_DIR does not exist:")
        print(ROOT_DIR)
        return

    if not ROOT_DIR.is_dir():
        print(f"[ERROR] ROOT_DIR is not a directory:")
        print(ROOT_DIR)
        return

    output_root = OUTPUT_ROOT
    output_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    # ------------------------------------------------------------
    # Counters
    # ------------------------------------------------------------

    total = 0
    skipped = 0
    processed = 0
    failed = 0

    # ------------------------------------------------------------
    # FULL RECURSIVE SEARCH
    # ------------------------------------------------------------

    print("=" * 70)
    print("Recursive AVI search")
    print("=" * 70)

    print(f"Input root : {ROOT_DIR}")
    print(f"Output root: {OUTPUT_ROOT}")
    print()

    for avi in ROOT_DIR.rglob("*.avi"):

        total += 1

        # --------------------------------------------------------
        # Skip files that should not be processed
        # --------------------------------------------------------

        if should_skip_input(
            avi,
            output_root,
        ):
            print(f"[SKIP input] {avi}")
            skipped += 1
            continue

        # --------------------------------------------------------
        # Find Day folder recursively from parent hierarchy
        # --------------------------------------------------------

        day_dir = find_day_folder(
            avi,
            ROOT_DIR,
        )

        if day_dir is None:
            print(
                f"[WARN] Could not find Day* folder for:"
                f"\n       {avi}"
            )

            skipped += 1
            continue

        day = day_dir.name

        # --------------------------------------------------------
        # Find mouse folder recursively from parent hierarchy
        # --------------------------------------------------------

        mouse_dir = find_mouse_folder(
            avi,
            ROOT_DIR,
        )

        if mouse_dir is None:
            print(
                f"[WARN] Could not find mouse folder for:"
                f"\n       {avi}"
            )

            skipped += 1
            continue

        mouse_id = find_mouse_id(
            mouse_dir.name
        )

        # --------------------------------------------------------
        # Determine behavior/camera tag
        # --------------------------------------------------------

        tag = detect_behavior_tag(avi)

        if tag is None:
            print(
                f"[WARN] Could not determine behavior tag for:"
                f"\n       {avi}"
            )

            skipped += 1
            continue

        # --------------------------------------------------------
        # Construct destination
        # --------------------------------------------------------
        #
        # Mirror the input directory tree underneath OUTPUT_ROOT
        # only as far as the detected Day* folder.
        #
        # This preserves optional grouping folders above the mouse,
        # but drops every raw-data folder below Day* (for example
        # "Arena Top View", "Arena Mouse View", "Face", etc.).
        #
        # Example:
        #
        #   ROOT_DIR/
        #       Group_A/
        #           20250817_mouse975826/
        #               Day01/
        #                   Arena Top View/
        #                       raw.avi
        #
        # becomes:
        #
        #   OUTPUT_ROOT/
        #       Group_A/
        #           20250817_mouse975826/
        #               Day01/
        #                   mouse975826_Day01_topView_comp.avi
        #
        try:
            mirrored_day = day_dir.relative_to(ROOT_DIR)
        except ValueError:
            print(
                f"[WARN] Could not construct mirrored output path for:"
                f"\n       {avi}"
            )
            skipped += 1
            continue

        out_dir = output_root / mirrored_day

        out_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        out_name = (
            f"{mouse_id}_"
            f"{day}_"
            f"{tag}_comp.avi"
        )

        out_path = out_dir / out_name

        # --------------------------------------------------------
        # Skip existing output
        # --------------------------------------------------------

        if output_exists(out_path):

            print(
                f"[SKIP existing] {out_path}"
            )

            skipped += 1
            continue

        # --------------------------------------------------------
        # Horizontal flip
        # --------------------------------------------------------

        # Anything contained somewhere under an "Arena" folder
        # will be horizontally flipped.
        flip_h = is_inside_arena_folder(avi)

        # --------------------------------------------------------
        # Display detected information
        # --------------------------------------------------------

        print()
        print("-" * 70)
        print(f"[SOURCE] {avi}")
        print(f"[MOUSE ] {mouse_dir.name}")
        print(f"[ID    ] {mouse_id}")
        print(f"[DAY   ] {day}")
        print(f"[TAG   ] {tag}")
        print(f"[FLIP  ] {flip_h}")
        print(f"[OUTPUT] {out_path}")

        # --------------------------------------------------------
        # Compress
        # --------------------------------------------------------

        success = compress_with_ffmpeg(
            src=avi,
            dst=out_path,
            tag=tag,
            flip_h=flip_h,
        )

        if success:
            processed += 1
        else:
            failed += 1

    # ------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------

    print()
    print("=" * 70)
    print("Summary")
    print("=" * 70)

    print(f"Total .avi files found : {total}")
    print(f"Skipped                : {skipped}")
    print(f"Compressed             : {processed}")
    print(f"Failed                 : {failed}")


if __name__ == "__main__":
    main()



Recursive AVI search
Input root : D:\trainingAggression_MR
Output root: D:\observationSessions_bento

[SKIP existing] D:\observationSessions_bento\20250817_mouse975826\Day01\mouse975826_Day01_mouseView_comp.avi
[SKIP existing] D:\observationSessions_bento\20250817_mouse975826\Day01\mouse975826_Day01_topView_comp.avi
[SKIP existing] D:\observationSessions_bento\20250817_mouse975826\Day01\mouse975826_Day01_face_comp.avi
[SKIP existing] D:\observationSessions_bento\20250817_mouse975826\Day01\mouse975826_Day01_hs_comp.avi
[SKIP existing] D:\observationSessions_bento\20250817_mouse975826\Day03\mouse975826_Day03_mouseView_comp.avi
[SKIP existing] D:\observationSessions_bento\20250817_mouse975826\Day03\mouse975826_Day03_topView_comp.avi
[SKIP existing] D:\observationSessions_bento\20250817_mouse975826\Day03\mouse975826_Day03_face_comp.avi
[SKIP existing] D:\observationSessions_bento\20250817_mouse975826\Day03\mouse975826_Day03_hs_comp.avi
[SKIP existing] D:\observationSessions_bento\20250817_

In [2]:
from pathlib import Path
import subprocess
import shlex
import time

# ---------------- Configuration ----------------

SRC = Path(
    r"C:\Users\stagk\Desktop\trainingAggression_MR\20250825_mouse1055494\Day01\behavioralData\Arena (Mouse View)\19-58-16.avi"
)

DST = Path(
    r"C:\Users\stagk\Desktop\TempECG"
    r"\20250825_mouse1055494\Day01"
    r"\mouse1055494_Day01_mouseView_comp.avi"
)

FFMPEG = r"C:\Users\stagk\Desktop\ffmpeg\bin\ffmpeg.exe"

PRESET = "slow"
CRF = 20
CUT_TO = "00:20:05"

PIX_FMT = "yuv420p"
TARGET_FPS = 50

OVERWRITE = True
DRY_RUN = False

# ------------------------------------------------


def main():

    if not SRC.exists():
        raise FileNotFoundError(f"Input video not found:\n{SRC}")

    if not Path(FFMPEG).exists():
        raise FileNotFoundError(f"ffmpeg not found:\n{FFMPEG}")

    DST.parent.mkdir(parents=True, exist_ok=True)

    # mouseView pipeline:
    # - force 50 fps
    # - horizontal flip
    # - H.264 / CRF 20
    # - yuv420p
    # - remove audio
    # - cut at 20:05

    vf = f"fps={TARGET_FPS},hflip"

    cmd = [
        FFMPEG,
        "-i", str(SRC),

        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(CRF),

        "-pix_fmt", PIX_FMT,

        "-vf", vf,
        "-vsync", "cfr",

        "-to", CUT_TO,

        "-an",
    ]

    if OVERWRITE:
        cmd.append("-y")

    cmd.append(str(DST))

    print("\nInput:")
    print(SRC)

    print("\nOutput:")
    print(DST)

    print("\nCommand:")
    print(" ".join(shlex.quote(x) for x in cmd))

    if DRY_RUN:
        print("\n[DRY RUN] Nothing executed.")
        return

    print("\nStarting compression...")

    start = time.time()

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
    )

    elapsed = time.time() - start

    if result.returncode != 0:
        print("\n[ERROR] ffmpeg failed:")
        print(result.stderr)
        raise RuntimeError("ffmpeg compression failed")

    print(
        f"\n[OK] Compression finished\n"
        f"Output : {DST}\n"
        f"CRF    : {CRF}\n"
        f"FPS    : {TARGET_FPS} CFR\n"
        f"Format : {PIX_FMT}\n"
        f"Flip   : horizontal\n"
        f"Cut    : {CUT_TO}\n"
        f"Audio  : removed\n"
        f"Time   : {elapsed / 60:.2f} min ({elapsed:.1f} s)"
    )


if __name__ == "__main__":
    main()


Input:
C:\Users\stagk\Desktop\trainingAggression_MR\20250825_mouse1055494\Day01\behavioralData\Arena (Mouse View)\19-58-16.avi

Output:
C:\Users\stagk\Desktop\TempECG\20250825_mouse1055494\Day01\mouse1055494_Day01_mouseView_comp.avi

Command:
'C:\Users\stagk\Desktop\ffmpeg\bin\ffmpeg.exe' -i 'C:\Users\stagk\Desktop\trainingAggression_MR\20250825_mouse1055494\Day01\behavioralData\Arena (Mouse View)\19-58-16.avi' -c:v libx264 -preset slow -crf 20 -pix_fmt yuv420p -vf fps=50,hflip -vsync cfr -to 00:20:05 -an -y 'C:\Users\stagk\Desktop\TempECG\20250825_mouse1055494\Day01\mouse1055494_Day01_mouseView_comp.avi'

Starting compression...

[OK] Compression finished
Output : C:\Users\stagk\Desktop\TempECG\20250825_mouse1055494\Day01\mouse1055494_Day01_mouseView_comp.avi
CRF    : 20
FPS    : 50 CFR
Format : yuv420p
Flip   : horizontal
Cut    : 00:20:05
Audio  : removed
Time   : 4.27 min (256.2 s)


In [4]:
from pathlib import Path
import subprocess
import shlex
import time

FFMPEG = r"C:\Users\stagk\Desktop\ffmpeg\bin\ffmpeg.exe"

VIDEO = Path(
    r"C:\Users\stagk\Desktop\TempECG"
    r"\20250825_mouse1055494\Day01"
    r"\mouse1055494_Day01_mouseView_comp.avi"
)

AUDIO = Path(
    r"C:\Users\stagk\Desktop\trainingAggression_MR"
    r"\20250825_mouse1055494\Day01\usvData"
    r"\20260825_mouse1055494_obs1.wav"
)

OUTPUT = Path(
    r"C:\Users\stagk\Desktop\TempECG"
    r"\20250825_mouse1055494\Day01"
    r"\mouse1055494_Day01_mouseView_comp_audio.avi"
)


def main():

    if not VIDEO.exists():
        raise FileNotFoundError(f"Video not found:\n{VIDEO}")

    if not AUDIO.exists():
        raise FileNotFoundError(f"Audio not found:\n{AUDIO}")

    OUTPUT.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        FFMPEG,

        # Inputs
        "-i", str(VIDEO),
        "-i", str(AUDIO),

        # Explicitly use:
        # video from input 0
        # audio from input 1
        "-map", "0:v:0",
        "-map", "1:a:0",

        # Do NOT recompress the video
        "-c:v", "copy",

        # Preserve the WAV audio exactly, including its
        # native 300 kHz sampling rate
        "-c:a", "copy",

        # Stop when the shorter stream ends
        "-shortest",

        "-y",
        str(OUTPUT),
    ]

    print("\nCommand:")
    print(" ".join(shlex.quote(x) for x in cmd))

    print("\nAdding audio...")
    start = time.time()

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
    )

    elapsed = time.time() - start

    if result.returncode != 0:
        print("\n[ERROR]")
        print(result.stderr)
        raise RuntimeError("ffmpeg failed")

    print("\n[OK]")
    print(f"Video : {VIDEO}")
    print(f"Audio : {AUDIO}")
    print(f"Output: {OUTPUT}")
    print(f"Time  : {elapsed:.1f} s")


if __name__ == "__main__":
    main()


Command:
'C:\Users\stagk\Desktop\ffmpeg\bin\ffmpeg.exe' -i 'C:\Users\stagk\Desktop\TempECG\20250825_mouse1055494\Day01\mouse1055494_Day01_mouseView_comp.avi' -i 'C:\Users\stagk\Desktop\trainingAggression_MR\20250825_mouse1055494\Day01\usvData\20260825_mouse1055494_obs1.wav' -map 0:v:0 -map 1:a:0 -c:v copy -c:a copy -shortest -y 'C:\Users\stagk\Desktop\TempECG\20250825_mouse1055494\Day01\mouse1055494_Day01_mouseView_comp_audio.avi'

Adding audio...

[OK]
Video : C:\Users\stagk\Desktop\TempECG\20250825_mouse1055494\Day01\mouse1055494_Day01_mouseView_comp.avi
Audio : C:\Users\stagk\Desktop\trainingAggression_MR\20250825_mouse1055494\Day01\usvData\20260825_mouse1055494_obs1.wav
Output: C:\Users\stagk\Desktop\TempECG\20250825_mouse1055494\Day01\mouse1055494_Day01_mouseView_comp_audio.avi
Time  : 1.5 s
